# Week 2 : Day 1 Text Splitters & Chunking Strategies

### Written by: Hafiza Mehak Arif

### Date: 15-June-2026


In [15]:
!pip install langchain-text-splitters



[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


## Loading the document


In [16]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("Artificial Intelligence AI Everything you need to know.pdf")
document = loader.load()
print("Document Successfully Loaded")



Document Successfully Loaded


In [17]:
print("Number of pages:", len(document))

print(document[0].page_content[:500])
print(document[0].metadata)

Number of pages: 7
Artificial Intelligence (AI): Everything you need to
know
clearias.com
/artificial-intelligence-ai
Artificial Intelligence (AI) is a hot word these days. In this post, we cover artificial
intelligence in detail – its meaning, scope, risks, ethics etc.
What is Artificial Intelligence (AI)?
To make it simple – Artificial Intelligence is intelligence exhibited by machines.
It is a branch of computer science which deals with creating computers or
machines as intelligent as human beings.
The term was
{'producer': 'Qt 4.8.6', 'creator': 'wkhtmltopdf 0.12.2.1', 'creationdate': '2020-10-30T10:49:06+00:00', 'title': 'Artificial Intelligence (AI): Everything you need to know', 'source': 'Artificial Intelligence AI Everything you need to know.pdf', 'total_pages': 7, 'page': 0, 'page_label': '1'}


### Splitting the Document


In [18]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

chunks = splitter.split_documents(document)
print(len(chunks))
print(chunks[0].page_content)
print(chunks[0].metadata)

29
Artificial Intelligence (AI): Everything you need to
know
clearias.com
/artificial-intelligence-ai
Artificial Intelligence (AI) is a hot word these days. In this post, we cover artificial
intelligence in detail – its meaning, scope, risks, ethics etc.
What is Artificial Intelligence (AI)?
To make it simple – Artificial Intelligence is intelligence exhibited by machines.
It is a branch of computer science which deals with creating computers or
machines as intelligent as human beings.
{'producer': 'Qt 4.8.6', 'creator': 'wkhtmltopdf 0.12.2.1', 'creationdate': '2020-10-30T10:49:06+00:00', 'title': 'Artificial Intelligence (AI): Everything you need to know', 'source': 'Artificial Intelligence AI Everything you need to know.pdf', 'total_pages': 7, 'page': 0, 'page_label': '1'}


In [19]:
text = """
Machine learning is a branch of artificial intelligence that enables computers to learn from data without being explicitly programmed.
Supervised learning uses labeled data to train models for prediction tasks such as classification and regression.
Unsupervised learning discovers hidden patterns in unlabeled data through techniques such as clustering and dimensionality reduction.
Deep learning is a subset of machine learning that uses neural networks with multiple layers to model complex relationships.
Convolutional neural networks are commonly used in computer vision tasks such as image classification and object detection.
Recurrent neural networks and transformers are widely applied in natural language processing.
Large language models are transformer-based systems trained on vast amounts of text data.
Retrieval-Augmented Generation combines retrieval systems with language models to provide grounded and contextually relevant responses.
Chunking strategies significantly influence retrieval quality in RAG systems.
Choosing an appropriate chunk size and overlap is essential for preserving semantic meaning and improving search performance.
""" 

In [20]:
from langchain_text_splitters import (
    RecursiveCharacterTextSplitter ,
    CharacterTextSplitter
)


In [21]:
char_splitter = CharacterTextSplitter(
    separator="",
    chunk_size=500,
    chunk_overlap=0
)

char_chunks = char_splitter.split_text(text)

print("Number of Character Chunks:", len(char_chunks))

for i, chunk in enumerate(char_chunks[:3]):
    print(f"\nChunk {i+1}")
    print(chunk)
    print("-" * 80)

Number of Character Chunks: 3

Chunk 1
Machine learning is a branch of artificial intelligence that enables computers to learn from data without being explicitly programmed.
Supervised learning uses labeled data to train models for prediction tasks such as classification and regression.
Unsupervised learning discovers hidden patterns in unlabeled data through techniques such as clustering and dimensionality reduction.
Deep learning is a subset of machine learning that uses neural networks with multiple layers to model complex relati
--------------------------------------------------------------------------------

Chunk 2
onships.
Convolutional neural networks are commonly used in computer vision tasks such as image classification and object detection.
Recurrent neural networks and transformers are widely applied in natural language processing.
Large language models are transformer-based systems trained on vast amounts of text data.
Retrieval-Augmented Generation combines retrieval syst

Findings:
CharacterTextSplitter produced fixed-size chunks but frequently broke sentences and occasionally split ideas across chunk boundaries. This reduced semantic coherence.

RecursiveCharacterTextSplitter preserved meaning more effectively by attempting to split using natural separators such as paragraphs and sentences. The 50-character overlap further improved context preservation between adjacent chunks.

Therefore, RecursiveCharacterTextSplitter is generally a better choice for RAG applications because it balances chunk consistency with semantic integrity.


# Task 2: Optimal Chunk Size Experiment


In [22]:
full_text = "\n".join(doc.page_content for doc in document)

In [23]:
chunk_sizes = [200, 500, 1000, 2000]
overlap = 50

In [24]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

In [25]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

results = {}

for size in chunk_sizes:

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=size,
        chunk_overlap=overlap
    )

    chunks = splitter.split_text(full_text)

    embeddings = embedding_model.encode(chunks)

    results[size] = {
        "chunks": chunks,
        "embeddings": embeddings
    }

    print(f"Chunk Size: {size}")
    print(f"Total Chunks: {len(chunks)}")
    print("-" * 50)

Chunk Size: 200
Total Chunks: 78
--------------------------------------------------
Chunk Size: 500
Total Chunks: 27
--------------------------------------------------
Chunk Size: 1000
Total Chunks: 13
--------------------------------------------------
Chunk Size: 2000
Total Chunks: 7
--------------------------------------------------


In [26]:
import numpy as np

for size in chunk_sizes:

    embeddings = results[size]["embeddings"]

    storage_bytes = embeddings.nbytes

    print(f"Chunk Size: {size}")
    print(f"Embedding Shape: {embeddings.shape}")
    print(f"Storage: {storage_bytes/1024:.2f} KB")
    print()

Chunk Size: 200
Embedding Shape: (78, 384)
Storage: 117.00 KB

Chunk Size: 500
Embedding Shape: (27, 384)
Storage: 40.50 KB

Chunk Size: 1000
Embedding Shape: (13, 384)
Storage: 19.50 KB

Chunk Size: 2000
Embedding Shape: (7, 384)
Storage: 10.50 KB



In [31]:
queries = [
    "What is machine learning?",
    "How are we going to the types of ML?",
    "What is AI ? Explian its scope."
]

from sklearn.metrics.pairwise import cosine_similarity

def retrieve(query, chunks, embeddings, top_k=3):

    query_embedding = embedding_model.encode([query])

    similarities = cosine_similarity(
        query_embedding,
        embeddings
    )[0]

    indices = similarities.argsort()[::-1][:top_k]

    return [(chunks[i], similarities[i]) for i in indices]

for size in chunk_sizes:

    print("="*60)
    print(f"Chunk Size: {size}")

    for query in queries:

        print(f"\nQuery: {query}")

        results_top = retrieve(
            query,
            results[size]["chunks"],
            results[size]["embeddings"]
        )

        best_chunk, score = results_top[0]

        print(f"Score: {score:.4f}")
        print(best_chunk[:250])
        print()

Chunk Size: 200

Query: What is machine learning?
Score: 0.6895
analysis.
Machine learning
: Field of study that gives computers the ability to learn
without being explicitly programmed.  
Deep learning
 is a subset of machine


Query: How are we going to the types of ML?
Score: 0.3413
3/7
Click here for more
@IAS_TUTORIAL
Healthcare Sector
: Machine learning is being used for faster, cheaper and more
accurate diagnosis and thus improving patient outcomes and reducing costs. For


Query: What is AI ? Explian its scope.
Score: 0.6594
intelligence in detail – its meaning, scope, risks, ethics etc.
What is Artificial Intelligence (AI)?
To make it simple – Artificial Intelligence is intelligence exhibited by machines.

Chunk Size: 500

Query: What is machine learning?
Score: 0.6769
patterns in data.
Machine vision
 is the science of making computers visualize by capturing and
analyzing visual information using a camera, analog-to-digital conversion, and
digital signal processing. It is oft

### Findings:

RAG quality depends heavily on choosing the right chunk size, and the optimal size.
